In [ ]:
import math
from dataclasses import dataclass
from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


# -----------------------------
# Config
# -----------------------------

@dataclass
class BARTConfig:
    vocab_size: int = 30000
    d_model: int = 512
    encoder_layers: int = 6
    decoder_layers: int = 6
    n_heads: int = 8
    d_ff: int = 2048
    dropout: float = 0.1
    max_position_embeddings: int = 512
    pad_token_id: int = 0


# -----------------------------
# Positional Embeddings (learned)
# -----------------------------

class LearnedPositionalEmbedding(nn.Module):
    def __init__(self, max_len: int, d_model: int):
        super().__init__()
        self.weight = nn.Embedding(max_len, d_model)

    def forward(self, seq_len: int, device=None):
        positions = torch.arange(0, seq_len, dtype=torch.long, device=device)
        return self.weight(positions)  # (seq_len, d_model)


# -----------------------------
# Multi-head Attention
# -----------------------------

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def _shape(self, x: torch.Tensor, seq_len: int, bsz: int):
        # (batch, seq_len, d_model) -> (batch, num_heads, seq_len, d_head)
        return x.view(bsz, seq_len, self.num_heads, self.d_head).transpose(1, 2)

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
        key_padding_mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        query, key, value: (batch, seq_len, d_model)
        attn_mask: (target_len, source_len) or (batch * num_heads, target_len, source_len)
        key_padding_mask: (batch, source_len) -> True for positions to mask
        """
        bsz, tgt_len, _ = query.size()
        src_len = key.size(1)

        q = self._shape(self.q_proj(query), tgt_len, bsz)
        k = self._shape(self.k_proj(key), src_len, bsz)
        v = self._shape(self.v_proj(value), src_len, bsz)

        # scaled dot-product attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)  # (b, h, tgt, src)

        if attn_mask is not None:
            # attn_mask: broadcast to (b, h, tgt, src)
            if attn_mask.dim() == 2:
                attn_mask = attn_mask.unsqueeze(0).unsqueeze(0)  # (1,1,tgt,src)
            scores = scores.masked_fill(attn_mask == 1, float('-inf'))

        if key_padding_mask is not None:
            # key_padding_mask: (b, src) -> (b,1,1,src)
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        attn_output = torch.matmul(attn_weights, v)  # (b, h, tgt, d_head)
        attn_output = attn_output.transpose(1, 2).contiguous().view(bsz, tgt_len, self.d_model)
        attn_output = self.out_proj(attn_output)
        return attn_output, attn_weights


# -----------------------------
# Feed-forward block
# -----------------------------

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


## Encoder & Decoder Layers


In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, config: BARTConfig):
        super().__init__()
        self.self_attn = MultiHeadAttention(config.d_model, config.n_heads, config.dropout)
        self.self_attn_layer_norm = nn.LayerNorm(config.d_model)
        self.ff = PositionwiseFeedForward(config.d_model, config.d_ff, config.dropout)
        self.ff_layer_norm = nn.LayerNorm(config.d_model)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, src_key_padding_mask=None):
        # Self-attention
        residual = x
        attn_output, _ = self.self_attn(x, x, x, key_padding_mask=src_key_padding_mask)
        x = residual + self.dropout(attn_output)
        x = self.self_attn_layer_norm(x)

        # Feed-forward
        residual = x
        ff_output = self.ff(x)
        x = residual + self.dropout(ff_output)
        x = self.ff_layer_norm(x)
        return x


class DecoderLayer(nn.Module):
    def __init__(self, config: BARTConfig):
        super().__init__()
        self.self_attn = MultiHeadAttention(config.d_model, config.n_heads, config.dropout)
        self.self_attn_layer_norm = nn.LayerNorm(config.d_model)

        self.cross_attn = MultiHeadAttention(config.d_model, config.n_heads, config.dropout)
        self.cross_attn_layer_norm = nn.LayerNorm(config.d_model)

        self.ff = PositionwiseFeedForward(config.d_model, config.d_ff, config.dropout)
        self.ff_layer_norm = nn.LayerNorm(config.d_model)

        self.dropout = nn.Dropout(config.dropout)

    def _generate_causal_mask(self, size: int, device=None):
        # mask[i, j] = 1 if j > i else 0  (upper triangular)
        mask = torch.triu(torch.ones(size, size, device=device), diagonal=1)
        return mask  # (size, size)

    def forward(
        self,
        x,
        encoder_out,
        tgt_key_padding_mask=None,
        memory_key_padding_mask=None,
    ):
        batch_size, tgt_len, _ = x.size()
        device = x.device

        # Causal self-attention
        residual = x
        causal_mask = self._generate_causal_mask(tgt_len, device=device)
        self_attn_out, _ = self.self_attn(
            query=x,
            key=x,
            value=x,
            attn_mask=causal_mask,
            key_padding_mask=tgt_key_padding_mask,
        )
        x = residual + self.dropout(self_attn_out)
        x = self.self_attn_layer_norm(x)

        # Cross-attention
        residual = x
        cross_attn_out, _ = self.cross_attn(
            query=x,
            key=encoder_out,
            value=encoder_out,
            key_padding_mask=memory_key_padding_mask,
        )
        x = residual + self.dropout(cross_attn_out)
        x = self.cross_attn_layer_norm(x)

        # Feed-forward
        residual = x
        ff_out = self.ff(x)
        x = residual + self.dropout(ff_out)
        x = self.ff_layer_norm(x)

        return x

## Encoder & Decoder Stacks

In [ ]:
class BARTEncoder(nn.Module):
    def __init__(self, config: BARTConfig):
        super().__init__()
        self.embed_tokens = nn.Embedding(config.vocab_size, config.d_model, padding_idx=config.pad_token_id)
        self.embed_positions = LearnedPositionalEmbedding(config.max_position_embeddings, config.d_model)
        self.layers = nn.ModuleList([EncoderLayer(config) for _ in range(config.encoder_layers)])
        self.layer_norm = nn.LayerNorm(config.d_model)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, src_tokens, src_key_padding_mask=None):
        """
        src_tokens: (batch, src_len)
        src_key_padding_mask: (batch, src_len) -> True where pad
        """
        device = src_tokens.device
        batch_size, src_len = src_tokens.size()

        x = self.embed_tokens(src_tokens)  # (b, src_len, d_model)
        pos_emb = self.embed_positions(src_len, device=device)  # (src_len, d_model)
        x = x + pos_emb.unsqueeze(0)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, src_key_padding_mask=src_key_padding_mask)

        x = self.layer_norm(x)
        return x


class BARTDecoder(nn.Module):
    def __init__(self, config: BARTConfig):
        super().__init__()
        self.embed_tokens = nn.Embedding(config.vocab_size, config.d_model, padding_idx=config.pad_token_id)
        self.embed_positions = LearnedPositionalEmbedding(config.max_position_embeddings, config.d_model)
        self.layers = nn.ModuleList([DecoderLayer(config) for _ in range(config.decoder_layers)])
        self.layer_norm = nn.LayerNorm(config.d_model)
        self.dropout = nn.Dropout(config.dropout)

    def forward(
        self,
        tgt_tokens,
        encoder_out,
        tgt_key_padding_mask=None,
        memory_key_padding_mask=None,
    ):
        """
        tgt_tokens: (batch, tgt_len)
        encoder_out: (batch, src_len, d_model)
        """
        device = tgt_tokens.device
        batch_size, tgt_len = tgt_tokens.size()

        x = self.embed_tokens(tgt_tokens)
        pos_emb = self.embed_positions(tgt_len, device=device)
        x = x + pos_emb.unsqueeze(0)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(
                x,
                encoder_out,
                tgt_key_padding_mask=tgt_key_padding_mask,
                memory_key_padding_mask=memory_key_padding_mask,
            )

        x = self.layer_norm(x)
        return x


## Full BART Seq2Seq Model

In [ ]:
class BARTModel(nn.Module):
    def __init__(self, config: BARTConfig):
        super().__init__()
        self.config = config
        self.encoder = BARTEncoder(config)
        self.decoder = BARTDecoder(config)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        # Tie weights
        self.lm_head.weight = self.decoder.embed_tokens.weight

    def forward(
        self,
        src_tokens,
        tgt_tokens,
        src_key_padding_mask=None,
        tgt_key_padding_mask=None,
    ):
        """
        Forward pass for training (teacher forcing).

        src_tokens: (batch, src_len)
        tgt_tokens: (batch, tgt_len)  - usually shifted right (BOS + target[:-1])
        """
        encoder_out = self.encoder(src_tokens, src_key_padding_mask=src_key_padding_mask)

        decoder_out = self.decoder(
            tgt_tokens,
            encoder_out,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )

        logits = self.lm_head(decoder_out)  # (batch, tgt_len, vocab_size)
        return logits

    @torch.no_grad()
    def generate(
        self,
        src_tokens,
        src_key_padding_mask=None,
        max_len: int = 50,
        bos_token_id: int = 2,
        eos_token_id: int = 3,
    ):
        """
        Very simple greedy decoding, just for demo.
        """
        device = src_tokens.device
        batch_size = src_tokens.size(0)

        encoder_out = self.encoder(src_tokens, src_key_padding_mask=src_key_padding_mask)

        # start with BOS
        generated = torch.full((batch_size, 1), bos_token_id, dtype=torch.long, device=device)

        finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for _ in range(max_len):
            decoder_out = self.decoder(
                generated,
                encoder_out,
                tgt_key_padding_mask=None,
                memory_key_padding_mask=src_key_padding_mask,
            )
            logits = self.lm_head(decoder_out)  # (b, t, vocab)
            next_token = logits[:, -1, :].argmax(-1)  # (b,)

            generated = torch.cat([generated, next_token.unsqueeze(1)], dim=1)

            finished = finished | (next_token == eos_token_id)
            if finished.all():
                break

        return generated
        

In [ ]:
if __name__ == "__main__":
    config = BARTConfig(
        vocab_size=100,  # toy vocab
        d_model=256,
        encoder_layers=3,
        decoder_layers=3,
        n_heads=8,
        d_ff=1024,
        max_position_embeddings=128,
        pad_token_id=0,
    )

    model = BARTModel(config)
    model.train()

    batch_size = 2
    src_len = 10
    tgt_len = 12

    src = torch.randint(4, config.vocab_size, (batch_size, src_len))
    tgt_inp = torch.randint(4, config.vocab_size, (batch_size, tgt_len))
    tgt_out = torch.randint(4, config.vocab_size, (batch_size, tgt_len))

    #-pad masking example (here: no pads)
    src_pad_mask = src.eq(config.pad_token_id)
    tgt_pad_mask = tgt_inp.eq(config.pad_token_id)

    logits = model(src, tgt_inp, src_key_padding_mask=src_pad_mask, tgt_key_padding_mask=tgt_pad_mask)
    loss = F.cross_entropy(logits.view(-1, config.vocab_size), tgt_out.view(-1), ignore_index=config.pad_token_id)

    print("logits shape:", logits.shape)
    print("loss:", loss.item())

    # switch to eval and try generate
    model.eval()
    with torch.no_grad():
        gen = model.generate(src, src_key_padding_mask=src_pad_mask, max_len=5)
        print("generated ids:", gen)
